In [1]:
def stima_tempo(points):
    import requests

    points = [{'lat': 45.5173, 'lon': 9.2074},  # Milano Bicocca
            {'lat': 45.4642, 'lon': 9.1900}]  # Milano Duomo

    coords = []
    coords.extend(
        f"{p['lon']},{p['lat']}"
        for p in points
        )

    coord_string=";".join(coords)

    url=(f"https://router.project-osrm.org/table/v1/driving/{coord_string}")

    data=requests.get(url).json()

    return round(data["durations"][0][1]/60, 2)

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
routing_tomtom.py — Calcolo dei tempi di percorrenza nel traffico (TomTom)
per lo scoring di AuraMed.

Sostituisce OSRM: fornisce il tempo di percorrenza *che tiene conto del
traffico in tempo reale*, che è ciò che serve alla componente A/T della
funzione di scoring Q.

ATTENZIONE alle coordinate:
    TomTom usa l'ordine  (latitudine, longitudine)  -> es. (45.6983, 9.6773)
    OSRM     usa l'ordine (longitudine, latitudine).
    Qui tutte le coordinate sono (lat, lon).

La API key NON è nel codice: viene letta dalla variabile d'ambiente
TOMTOM_API_KEY. Impostala con:
    export TOMTOM_API_KEY="la_tua_chiave"        (Linux/Mac)
    setx  TOMTOM_API_KEY "la_tua_chiave"         (Windows)
oppure mettila in un file .env (fuori dal repo, in .gitignore).
"""

from __future__ import annotations

import os
import time
from dataclasses import dataclass, field
from typing import Iterable, Optional

import requests

# ---------------------------------------------------------------------------
# CONFIGURAZIONE
# ---------------------------------------------------------------------------

# Endpoint legacy Routing API v1 (stabile, ampiamente documentato).
# In alternativa Orbis:
#   https://api.tomtom.com/maps/orbis/routing/calculateRoute/...  (header TomTom-Api-Version:2)
# Il legacy funziona con la stessa key ed è più semplice per iniziare.
BASE_URL = "https://api.tomtom.com/routing/1/calculateRoute"

DEFAULT_TIMEOUT = 10  # secondi per richiesta HTTP

os.environ["TOMTOM_API_KEY"] = "5xPKjOqL3vYxZgLczBoICnHLvKICgYiD"

def _get_api_key(api_key: Optional[str]) -> str:
    key = api_key or os.environ.get("TOMTOM_API_KEY")
    if not key:
        raise RuntimeError(
            "API key TomTom mancante. Imposta la variabile d'ambiente "
            "TOMTOM_API_KEY oppure passa api_key=... alla funzione."
        )
    return key


# ---------------------------------------------------------------------------
# STRUTTURA DEL RISULTATO
# ---------------------------------------------------------------------------

@dataclass
class RisultatoPercorso:
    """Riepilogo di un singolo percorso origine -> destinazione."""
    tempo_traffico_s: int              # tempo stimato CON traffico attuale (best estimate)
    tempo_senza_traffico_s: Optional[int]  # tempo senza alcun traffico
    ritardo_traffico_s: Optional[int]  # ritardo dovuto al traffico
    distanza_m: int                    # distanza in metri
    geometria: list = field(default_factory=list)  # lista di (lat, lon), vuota se non richiesta

    @property
    def tempo_traffico_min(self) -> float:
        return round(self.tempo_traffico_s / 60.0, 1)

    @property
    def distanza_km(self) -> float:
        return round(self.distanza_m / 1000.0, 2)

    @property
    def ritardo_traffico_min(self) -> Optional[float]:
        if self.ritardo_traffico_s is None:
            return None
        return round(self.ritardo_traffico_s / 60.0, 1)


# ---------------------------------------------------------------------------
# CHIAMATA DI ROUTING
# ---------------------------------------------------------------------------

def calcola_percorso(
    origine: tuple[float, float],
    destinazione: tuple[float, float],
    api_key: Optional[str] = None,
    con_geometria: bool = False,
    timeout: int = DEFAULT_TIMEOUT,
) -> RisultatoPercorso:
    """Calcola un singolo percorso origine -> destinazione (auto, nel traffico).

    Args:
        origine:      (lat, lon) del punto di partenza (posizione utente).
        destinazione: (lat, lon) dell'ospedale.
        con_geometria: se True include la lista di punti del tracciato
                       (utile per disegnarlo sulla mappa).

    Returns:
        RisultatoPercorso con tempi (con/senza traffico), distanza ed
        eventualmente la geometria.
    """
    key = _get_api_key(api_key)

    # NB: lat,lon nell'ordine richiesto da TomTom
    loc = f"{origine[0]},{origine[1]}:{destinazione[0]},{destinazione[1]}"

    params = {
        "key": key,
        "traffic": "true",              # considera il traffico in tempo reale
        "travelMode": "car",
        "routeType": "fastest",
        "computeTravelTimeFor": "all",  # restituisce anche il tempo senza traffico
        "routeRepresentation": "polyline" if con_geometria else "summaryOnly",
    }

    resp = requests.get(f"{BASE_URL}/{loc}/json", params=params, timeout=timeout)
    resp.raise_for_status()
    data = resp.json()

    route = data["routes"][0]
    s = route["summary"]

    geometria: list = []
    if con_geometria:
        # I punti sono dentro le "legs"; li concateno in un'unica lista (lat, lon)
        for leg in route.get("legs", []):
            for p in leg.get("points", []):
                geometria.append((p["latitude"], p["longitude"]))

    return RisultatoPercorso(
        tempo_traffico_s=s["travelTimeInSeconds"],
        tempo_senza_traffico_s=s.get("noTrafficTravelTimeInSeconds"),
        ritardo_traffico_s=s.get("trafficDelayInSeconds"),
        distanza_m=s["lengthInMeters"],
        geometria=geometria,
    )


# ---------------------------------------------------------------------------
# INTERFACCIA PROVIDER-AGNOSTICA (per lo scoring)
# ---------------------------------------------------------------------------

def get_tempo_percorrenza_minuti(
    origine: tuple[float, float],
    destinazione: tuple[float, float],
    api_key: Optional[str] = None,
) -> float:
    """Interfaccia unica usata dalla funzione di scoring.

    Restituisce SOLO il tempo di percorrenza nel traffico, in minuti.
    Se un domani cambi provider (o torni a OSRM), riscrivi solo questa
    funzione: la funzione di scoring Q non va toccata.
    """
    r = calcola_percorso(origine, destinazione, api_key=api_key, con_geometria=False)
    return r.tempo_traffico_min


# ---------------------------------------------------------------------------
# 1 ORIGINE -> N OSPEDALI
# ---------------------------------------------------------------------------

def tempi_ospedali(
    origine: tuple[float, float],
    ospedali: Iterable[dict],
    api_key: Optional[str] = None,
    con_geometria: bool = False,
    pausa_s: float = 0.0,
) -> list[dict]:
    """Calcola i tempi nel traffico dall'utente a ogni ospedale.

    Args:
        origine:  (lat, lon) dell'utente.
        ospedali: iterabile di dict con almeno le chiavi:
                  {"nome": str, "lat": float, "lon": float}
        con_geometria: se True aggiunge il tracciato di ogni percorso
                       (serve per la mappa).
        pausa_s:  eventuale pausa tra le chiamate (se ti avvicini ai limiti QPS).

    Returns:
        Lista di dict, uno per ospedale, ordinata per tempo crescente:
        {
            "nome", "lat", "lon",
            "tempo_min", "distanza_km", "ritardo_min",
            "geometria" (se richiesta),
            "errore" (presente solo se la chiamata è fallita)
        }
    """
    key = _get_api_key(api_key)
    risultati: list[dict] = []

    for osp in ospedali:
        dest = (osp["lat"], osp["lon"])
        base = {"nome": osp.get("nome"), "lat": osp["lat"], "lon": osp["lon"]}
        try:
            r = calcola_percorso(origine, dest, api_key=key, con_geometria=con_geometria)
            base.update({
                "tempo_min": r.tempo_traffico_min,
                "distanza_km": r.distanza_km,
                "ritardo_min": r.ritardo_traffico_min,
            })
            if con_geometria:
                base["geometria"] = r.geometria
        except Exception as e:
            # Una destinazione che fallisce non deve interrompere le altre
            base.update({"tempo_min": None, "distanza_km": None,
                         "ritardo_min": None, "errore": str(e)})
        risultati.append(base)
        if pausa_s:
            time.sleep(pausa_s)

    # Ordino per tempo (gli errori, con tempo None, vanno in fondo)
    risultati.sort(key=lambda x: (x["tempo_min"] is None, x["tempo_min"]))
    return risultati


# ---------------------------------------------------------------------------
# ESEMPIO D'USO
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    # Utente a Bergamo (esempio)
    utente = (45.6983, 9.6773)

    # Esempio di ospedali (lat, lon fittizie a scopo dimostrativo)
    ospedali_demo = [
        {"nome": "Papa Giovanni XXIII (Bergamo)", "lat": 45.6960, "lon": 9.6420},
        {"nome": "Bolognini (Seriate)",           "lat": 45.6870, "lon": 9.7230},
        {"nome": "Pesenti Fenaroli (Alzano)",     "lat": 45.7310, "lon": 9.7290},
    ]

    esiti = tempi_ospedali(utente, ospedali_demo, con_geometria=False)
    print("Ospedali ordinati per tempo nel traffico:\n")
    for e in esiti:
        if e.get("errore"):
            print(f"  ! {e['nome']}: errore ({e['errore']})")
        else:
            r = f" (+{e['ritardo_min']} min traffico)" if e['ritardo_min'] else ""
            print(f"  - {e['nome']}: {e['tempo_min']} min, {e['distanza_km']} km{r}")


Ospedali ordinati per tempo nel traffico:

  - Papa Giovanni XXIII (Bergamo): 12.2 min, 4.49 km
  - Pesenti Fenaroli (Alzano): 13.6 min, 6.89 km
  - Bolognini (Seriate): 14.5 min, 5.03 km
